<a href="https://colab.research.google.com/github/sstanishk/tanishk-codeboosters-2026/blob/main/Day7/Day7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install chromadb sentence-transformers -q
print("Installation Successful")

Installation Successful


In [18]:
import pandas as pd
import numpy as np
import chromadb

from sentence_transformers import SentenceTransformer

print("All libraries imported successfully")
print(f"ChromaDB version: {chromadb.__version__}")

All libraries imported successfully
ChromaDB version: 1.5.9


In [19]:
documents={
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automabile",
    "SQL is used to query databases",
    "Machine learning trains models on data"
}

query_keyword="vehicle"
print("="*60)
print(f'Keyowrd: {query_keyword}')
print("="*60)

for i,doc in enumerate(documents):
    if query_keyword.lower() in doc.lower():
        print(f' found Document {i}: {doc}')
    else:
        print(f'missed Document {i}: {doc}')

Keyowrd: vehicle
missed Document 0: Machine learning trains models on data
missed Document 1: ETL is used to clean and transform data
missed Document 2: Cars and trucks are popular automabile
 found Document 3: A vehicle is a mode of transportation
missed Document 4: SQL is used to query databases


In [20]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Done")
print(f'{model.get_sentence_embedding_dimension()}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done
384


/tmp/ipykernel_600/4169060353.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'{model.get_sentence_embedding_dimension()}')


In [21]:
sentence="ELT is used to clean and tranform data"
embedding=model.encode(sentence)
print(f'input sentence: {sentence}')
print(f'embedding shape: {embedding.shape}')
print(f'embedding type :{type(embedding)}')
print(f'first 10 numbers: {embedding[:10].round(4)}')
print(f'Min value: {embedding.min():.4f}')
print(f'Max value: {embedding.max():.4f}')

input sentence: ELT is used to clean and tranform data
embedding shape: (384,)
embedding type :<class 'numpy.ndarray'>
first 10 numbers: [-0.0539  0.0229  0.0072 -0.0394  0.1002 -0.0829  0.0457  0.0407  0.03
 -0.0052]
Min value: -0.1657
Max value: 0.1578


In [22]:
from sentence_transformers import SentenceTransformer, util

documents = [
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automabile",
    "SQL is used to query databases",
    "Machine learning trains models on data",
    "GT car made in Germany"
]

query_keyword = "vehicle"

print("=" * 60)
print(f'Keyword: {query_keyword}')
print("=" * 60)

model = SentenceTransformer("all-MiniLM-L6-v2")

query_embedding = model.encode(query_keyword, convert_to_tensor=True)

for i, doc in enumerate(documents):
    doc_embedding = model.encode(doc, convert_to_tensor=True)

    score = util.cos_sim(query_embedding, doc_embedding).item()

    if score > 0.3:   # threshold
        print(f'Found Document {i}: {doc} (Score: {score:.4f})')
    else:
        print(f'Missed Document {i}: {doc} (Score: {score:.4f})')

Keyword: vehicle


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Missed Document 0: ETL is used to clean and transform data (Score: 0.0857)
Found Document 1: A vehicle is a mode of transportation (Score: 0.7259)
Found Document 2: Cars and trucks are popular automabile (Score: 0.5378)
Missed Document 3: SQL is used to query databases (Score: 0.1030)
Missed Document 4: Machine learning trains models on data (Score: 0.2016)
Found Document 5: GT car made in Germany (Score: 0.4453)


In [23]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("demo_notes")
print("chromaDB client created (in-memory mode)")
print(f"collection name: demo_notes")
print(f"Document is collection: {collection.count()}")

chromaDB client created (in-memory mode)
collection name: demo_notes
Document is collection: 0


In [24]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection("my_collection")
print("Done")
print(f'Collection name: {collection.name}')
print(f'Collection id: {collection.id}')
print(f'Collection count: {collection.count()}')

sample_doc=[
    "ETL is used to clean and transform data",
    "A vehicle is a mode of transportation",
    "Cars and trucks are popular automabile",
    "SQL is used to query databases",
    "Machine learning trains models on data",

]

sample_ids=["doc001","doc002","doc003","doc004","doc005"]


sample_metadata = [
    {"category": "Data Engineering", "topic": "ETL"},
    {"category": "Transportation", "topic": "Vehicle"},
    {"category": "Transportation", "topic": "Automobile"},
    {"category": "Database", "topic": "SQL"},
    {"category": "Artificial Intelligence", "topic": "Machine Learning"}
]

collection.add(
    documents=sample_doc,
    ids=sample_ids,
    metadatas=sample_metadata
)

print(f'Documents added complete')
print(f'total documents in collection: {collection.count()}')

Done
Collection name: my_collection
Collection id: 7a846e97-5f76-4eb5-8cb7-719a4b6d8e9b
Collection count: 5
Documents added complete
total documents in collection: 5


In [25]:
query = "How do I clean and prepare data?"

results = collection.query(
    query_texts=query,
    n_results=3
)
print("RESULT KEYS AVAILABLE:")
print(list(results.keys()))

RESULT KEYS AVAILABLE:
['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances']


In [26]:
#Display
print(f'Query:{query}')
print("="*60)
print()

matched_docs=results["documents"][0]
matched_ids=results["ids"][0]
matched_metadatas=results["metadatas"][0]
matched_distances=results["distances"][0]

for rank,(doc,id,metadata,distance) in enumerate(zip(matched_docs,matched_ids,matched_metadatas,matched_distances)):
    print(f'Rank:{rank+1}')
    print(f'Document:{doc}')
    print(f'ID:{id}')
    print(f'Metadata:{metadata}')
    print(f'Distance:{distance}')
    print("="*60)

Query:How do I clean and prepare data?

Rank:1
Document:ETL is used to clean and transform data
ID:doc001
Metadata:{'topic': 'ETL', 'category': 'Data Engineering'}
Distance:1.1972317695617676
Rank:2
Document:SQL is used to query databases
ID:doc004
Metadata:{'category': 'Database', 'topic': 'SQL'}
Distance:1.5368705987930298
Rank:3
Document:Machine learning trains models on data
ID:doc005
Metadata:{'category': 'Artificial Intelligence', 'topic': 'Machine Learning'}
Distance:1.7308849096298218


In [27]:
filtered_results = collection.query(
    query_texts=[query],
    n_results=3,
    where={"subject": "Machine Learning"}
)
print(f"FILTERED QUERY: '{query}'")
print("Filter: Only Machine Learning Documents")
print("="*60)
for rank, (doc, dist, meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['distances'][0],
    filtered_results['metadatas'][0]
), start=1):
    print(f"Rank {rank}| Distance:{dist:.4f} | Subject:{meta['subject']}")
    print(f"{doc}")
    print()
print("Notice: Only ML documents appear, even though ETL and pandas might be related")

FILTERED QUERY: 'How do I clean and prepare data?'
Filter: Only Machine Learning Documents
Notice: Only ML documents appear, even though ETL and pandas might be related
